# Notebook Prétraitement données EEG

Notebook pour explorer les données BIDS, appliquer un pipeline de nettoyage complet et factoriser le flux de travail en fonctions réutilisables.

### Contexte et références

Ce notebook prolonge le mobilise des notions introduites dans :
- Module 0 — gestion de l'environnement (fichiers, `conda`, `pip`)
- Module 3 — manipulation Python adaptée aux données M/EEG
- Module 4 — chargement et inspection des données MNE
- Module 5 — prétraitement (filtrage, AutoReject, ICA)

Chaque section annonce le type de bloc :
- **Type 1** : méthode de base prête à exécuter
- **Type 2** : bloc à compléter / faire des ajustements
- **Type 3** : exploration optionnelle/ouverte

> Informations: les données sont organisées en BIDS. Consultez la documentation [MNE-BIDS](https://mne.tools/mne-bids/stable/index.html) pour plus de détails.

Instructions pratiques — préparation des données et de l'environnement
- Téléchargez les données depuis le lien fourni (voir consigne du cours) et placez le dossier nommé `tasks` au même niveau que ce notebook (même dossier parent). Exemple de structure attendue :
    - project_folder/
        - 01_preprocessing_notebook.ipynb
        - tasks/

N'oublier pas de créer et activer un environnement virtuel (local / VSCode) comme vue deja en cours (Module 0 et 2 pour reference)
Pour ceux qui utilise VSCode : sélectionnez l'interpréteur du venv via "Python: Select Interpreter". 

Si vous travaillez sur Colab
- Soit téléversez le dossier `tasks` via l'interface "Files" de Colab,
- soit montez Google Drive et copiez le dossier :
    ```python
    from google.colab import drive
    drive.mount('/content/drive')
    # puis copier / lier votre dossier tasks dans /content
    ```

Installer les dépendances requises
- Exemple de commande pour installer les librairies utilisées dans ce notebook :
    ```
    pip install mne mne-bids autoreject mne-icalabel matplotlib pandas numpy
    ```
- Alternative : placez ces lignes dans un `requirements.txt` et lancez `pip install -r requirements.txt`.
- Remarque : dans un notebook, préférez `!pip install ...` pour vous assurer que l'installation cible l'interpréteur du kernel courant.

Vérification finale
- Après activation / sélection du kernel, exécutez une cellule courte pour confirmer les imports :
    ```python
    import mne, mne_bids, autoreject, mne_icalabel, numpy, pandas, matplotlib
    print('OK')
    ```
- Lancez ensuite la cellule qui définit `root_bids` et vérifiez que `tasks` (ou le dataset) est détecté.

### Instructions pour exécuter ce notebook

- Si vous n'avez pas encore cloné le dépôt du cours (PSY2007D2025-Cours-UdeM), ouvrez un terminal et collez/exécutez la commande suivante :
    ```bash  
    git clone https://github.com/BabaSanfour/PSY2007D2025-Cours-UdeM
    ```

- Si vous avez déjà cloné le repo et voulez récupérer les dernières modifications :
    ```bash
    cd PSY2007D2025
    git pull
    ```

- Si vous préférez une interface graphique :
  - GitHub Desktop : File → Clone repository → entrez l'URL.
  - VSCode : View → SCM → Clone Repository, ou utilisez l'icône Source Control.
  - Sur GitHub web : Code → Download ZIP (dézippez ensuite localement).

- Pire des cas — pas d'accès à git : copiez le notebook cellule par cellule dans un nouveau notebook vide.

Cloner et ouvrir directement dans Colab
- Clonage simple depuis Colab (dans une cellule) :
    ```bash
    !git clone https://github.com/BabaSanfour/PSY2007D2025-Cours-UdeM
    !ls -la
    ```

Conseil important — travaillez sur une copie
- Avant de modifier ce notebook partagé, créez et travaillez sur une copie :
  - Jupyter Notebook : File → Make a Copy... → renommez en `01_preprocessing_notebook_COPIE.ipynb`
  - Colab : File → Save a copy in Drive → renommez en `01_preprocessing_notebook_COPIE.ipynb`
- Travailler sur une copie évite d'écraser l'original et facilite les comparaisons / pull requests.

## 1. Explorer et visualiser les fichiers BIDS

Objectifs : comprendre la structure du dossier, charger un enregistrement et inspecter le signal brut.

### Bloc Type 1 — Méthode de base
Initialisation des bibliothèques et paramètres globaux (chemins BIDS, fréquence secteur, configuration graphique).

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from autoreject import AutoReject
from mne.preprocessing import ICA
from mne_bids import BIDSPath, read_raw_bids
from mne_icalabel.iclabel import iclabel_label_components

warnings.filterwarnings('ignore', category=RuntimeWarning)
mne.set_log_level('INFO')

# Renseignez ici la racine du dossier BIDS a utiliser
#exp tasks/fivepoint/bids/
root_bids = Path('tasks/fivepoint/bids/')
if not root_bids.exists():
    print('Attention: le dossier BIDS indique est introuvable. Mettez a jour `root_bids`.')
else:
    print('Dossier BIDS detecte:', root_bids.resolve())


### Bloc Type 2 — À vous de compléter
Énumérez les sessions et fichiers disponibles. Inspirez-vous du Module 0 (parcours de dossiers) et de la méthode `BIDSPath`.

In [ ]:
# Bloc Type 2 — À vous de compléter
# TODO: renseignez la liste des sujets et la session à traiter si la détection automatique ne convient pas.
# Astuce: utilisez Path.glob ou BIDSPath pour parcourir la structure BIDS (sub-*, ses-*, etc.).
subjects = ['A_COMPLETER']
session = 'A_COMPLETER'
task = 'fivepoint'
run = '01'

if not root_bids.exists():
    raise FileNotFoundError('root_bids introuvable. Mettez a jour la cellule d"initialisation (root_bids).')

# Détection automatique des sujets si l'utilisateur n'a pas renseigné `subjects`
if subjects == ['A_COMPLETER']:
    detected = sorted([p.name for p in root_bids.glob('sub-*') if p.is_dir()])
    if detected:
        # retire le préfixe 'sub-' pour correspondre à l'usage attendu ailleurs
        subjects = [s.replace('sub-', '') for s in detected]
        print('Sujets détectés automatiquement :', subjects)
    else:
        raise ValueError('Aucun sujet détecté dans root_bids. Renseignez `subjects` manuellement.')

# Détection simple de session (optionnelle) pour le premier sujet
if session == 'A_COMPLETER':
    subj_dir = root_bids / f'sub-{subjects[0]}'
    if subj_dir.exists():
        ses_dirs = sorted([p.name for p in subj_dir.glob('ses-*') if p.is_dir()])
        if ses_dirs:
            session = ses_dirs[0].replace('ses-', '')
            print('Session détectée automatiquement pour', subjects[0], ':', session)
        else:
            # pas de sessions explicites dans BIDS -> on utilise None (BIDSPath accepte None)
            session = None
            print('Aucune session détectée (structure sans ses-*) — session définie à None.')
    else:
        raise FileNotFoundError(f"Répertoire du sujet introuvable: {subj_dir}")

print('Sujets finaux :', subjects)
print('Session finale :', session)
print('Task :', task, 'Run :', run)


### Bloc Type 1 — Méthode de base
Chargement d'un enregistrement brut via `read_raw_bids` et inspection des métadonnées principales.

In [ ]:
if 'subjects' not in globals():
    raise NameError('Definissez `subjects` dans le bloc precedent.')

if subjects == ['A_COMPLETER'] or session == 'A_COMPLETER':
    raise ValueError('Remplissez les informations BIDS avant de continuer.')

bids_path = BIDSPath(root=root_bids, subject=subjects[0], session=session,
                     task=task, run=run, suffix='eeg', extension='.vhdr')
print('Lecture du fichier:', bids_path.fpath)
raw = read_raw_bids(bids_path)
raw.load_data()  # charge les données en mémoire
print('Données chargées avec succès:')
raw.info


In [ ]:
# Type 1 — Supprimer le canal 'Pz' (si présent) et mettre à jour la liste `canaux`
if 'Pz' in raw.ch_names:
    raw.drop_channels(['Pz'])
    print("Canal 'Pz' supprimé de `raw`.")
else:
    print("Canal 'Pz' absent — rien à faire.")

# Mettre à jour la sélection de canaux si nécessaire
if 'canaux' in globals():
    canaux = [ch for ch in canaux if ch != 'Pz']
    print('Liste `canaux` mise à jour :', canaux)

### Bloc Type 2 — À vous de compléter
Choisissez les canaux à afficher et ajustez la durée/offset d'affichage (`raw.plot`). Révisez les notions du Module 4.

In [ ]:
# TODO: parametrez les canaux et la duree de visualisation
canaux = ['P3', 'C3', 'P4']  # remplacez par une selection pertinente, raw.chn_names peut aider
duree = 60        # ex. 30, 60 ... (secondes)
debut = 200        # ex. 0 ou 60

if duree == 'A_COMPLETER' or debut == 'A_COMPLETER':
    raise ValueError('Definissez `duree` et `debut` avant de lancer ce bloc.')

fig = raw.plot(picks=canaux, duration=float(duree), start=float(debut),
               scalings='auto', title='Signal brut Mario', show=False)
plt.show()


### Bloc Type 3 — Pour aller plus loin
Comparez rapidement PSD ou topographies de bruit pour comprendre la qualité du signal. Appuyez-vous sur le Module 6.

In [ ]:
raw.compute_psd(picks='eeg').plot()
# en utilisant la documentation de MNE, essayez de personnaliser le tracé (ex. fmin, fmax, etc.)

## 2. Pipeline complet de nettoyage et de prétraitement

Objectifs : filtrage, AutoReject, ICA, ICLabel, deuxième passe et préparer les données pour des analyses ultérieures.

### Étape 2.1 — Filtrage et référence
Nous appliquons un filtre notch, un bandpass et une re-référence moyenne.

#### Bloc Type 1 — Méthode de base
Application des paramètres standards (60 Hz, 1-40 Hz).

In [ ]:
raw_filt = raw.copy()
raw_filt.notch_filter(freqs=[60])
raw_filt.filter(l_freq=1.0, h_freq=40.0, fir_design='firwin')
print('Filtrage')


#### Bloc Type 2 — À vous de compléter
Adaptez le filtrage (ex. 50 Hz pour l'Europe, bande plus large) et essayez une référence différente (`'REST'`, canal mastoïde, etc.). Consultez `Raw.filter`, `Raw.notch_filter`, `Raw.set_eeg_reference`.

In [ ]:
# TODO: personnalisez les parametres de filtrage et la reference
frequences_notch = [50, 100]  # ex. [50] ou [50, 100]
l_freq = 0.5              # ex. 0.5
h_freq = 45                # ex. 45

if 'A_COMPLETER' in [str(frequences_notch), str(l_freq), str(h_freq)]:
    raise ValueError('Remplissez les parametres avant d\'executer ce bloc.')

raw_custom = raw.copy()
raw_custom.notch_filter(freqs=[float(f) for f in frequences_notch])
raw_custom.filter(l_freq=float(l_freq), h_freq=float(h_freq), fir_design='firwin')
raw_custom


#### Bloc Type 3 — Pour aller plus loin
Visualisez l'impact du filtrage via des figures avant/après (traces, PSD, topomaps).

In [ ]:
raw.compute_psd(picks='eeg').plot()

### Étape 2.2 — Détection automatique des artéfacts (AutoReject)
Nous utilisons AutoReject pour rejeter ou réparer les epochs bruitées.

#### Bloc Type 1 — Méthode de base
Création d'epochs basées sur les annotations BIDS (`events.tsv`) et exécution d'AutoReject avec paramètres par défaut.

In [ ]:
# Appliquer le montage 10-20 avant la detection des events / AutoReject
montage_1020 = mne.channels.make_standard_montage('standard_1020')
raw_filt.set_montage(montage_1020, on_missing='warn')
print("Montage 10-20 applique (on_missing='warn').")

events, event_id = mne.events_from_annotations(raw_filt)
print("Nombre d'evenements detectes:", len(events))

epochs = mne.Epochs(
    raw_filt,
    events=events,
    event_id=event_id,
    tmin=-0.2,
    tmax=0.8,
    baseline=(None, 0),
    preload=True,
)
print(epochs)

# Conserver une copie du signal filtre avant ICA pour la reconstruction finale
raw_pre_ica = raw_filt.copy()

# Reutiliser l'instance existante si presente, sinon en creer une nouvelle
auto_reject = globals().get('auto_reject', AutoReject(random_state=42, verbose='tqdm'))
auto_reject.fit(epochs)
reject_log = auto_reject.get_reject_log(epochs)

# Masque des epochs juges valides
good_mask = ~reject_log.bad_epochs
print(f"Total epochs: {len(epochs)}, rejetes par AutoReject: {reject_log.bad_epochs.sum()}")

epochs_clean = epochs[good_mask].copy()
print(epochs_clean)


#### Bloc Type 2 — À vous de compléter
Ajustez les paramètres d'AutoReject (`cv`, `n_interpolate`, `consensus`) et la fenêtre temporelle. Voir Module 5 et la documentation AutoReject.

In [ ]:
# TODO: affiner AutoReject
cv = 'A_COMPLETER'                # ex. 10
n_interpolate = ['A_COMPLETER']   # ex. [1, 4, 8]
consensus = ['A_COMPLETER']       # ex. [0.8, 0.9]
tmin = 'A_COMPLETER'             # ex. -0.1
tmax = 'A_COMPLETER'             # ex. 0.6

if 'A_COMPLETER' in [str(cv), str(tmin), str(tmax)] or 'A_COMPLETER' in [str(x) for x in n_interpolate] or 'A_COMPLETER' in [str(x) for x in consensus]:
    raise ValueError('Remplissez tous les parametres avant d'executer ce bloc.')

epochs_tuned = mne.Epochs(raw_filt, events=events, event_id=event_id,
                           tmin=float(tmin), tmax=float(tmax), baseline=(None, 0), preload=True)

auto_reject_tuned = AutoReject(cv=int(cv), n_interpolate=[int(x) for x in n_interpolate],
                               consensus=[float(x) for x in consensus], random_state=42, verbose='tqdm')
epochs_tuned_clean, reject_log_tuned = auto_reject_tuned.fit_transform(epochs_tuned, return_log=True)
reject_log_tuned.plot('horizontal')


#### Bloc Type 3 — Pour aller plus loin
Analysez visuellement les epochs rejetées, comparez plusieurs configurations, documentez vos choix.

In [ ]:
fig = reject_log.plot('vertical')
fig.suptitle('Suivi des rejets AutoReject (configuration de base)')
plt.show()


### Étape 2.3 — ICA + ICLabel
Objectifs : identifier les composantes artefact et les exclure du signal.

#### Bloc Type 1 — Méthode de base
ICA sur les epochs nettoyées, application d'ICLabel et exclusion automatique des composantes oeil/muscle.

In [ ]:
# Fit ICA uniquement sur les epochs conserves par AutoReject
ica = ICA(n_components=18, method='fastica', random_state=97)
ica.fit(epochs_clean)

# 3. Classifier les composantes avec ICLabel et identifier celles liees aux yeux
labels = iclabel_label_components(epochs_clean, ica)
classes = np.array(labels['labels'])
probabilities = labels.get('probabilities', [])
class_names = labels.get('classes', ['brain', 'muscle', 'eye', 'heart', 'line_noise', 'channel_noise', 'other'])

if isinstance(probabilities, list):
    probas = np.array(probabilities)
else:
    probas = np.asarray(probabilities)

if probas.ndim != 2 or probas.shape[1] != len(class_names):
    probas = np.zeros((len(classes), len(class_names)))
    for idx, label in enumerate(classes):
        if label in class_names:
            probas[idx, class_names.index(label)] = 1.0

index_eye = class_names.index('eye')
eye_scores = probas[:, index_eye]

eye_components = [idx for idx, label in enumerate(classes) if label == 'eye' and eye_scores[idx] >= 0.7]
print('Scores ICLabel (oeil):')
for idx, (label, score) in enumerate(zip(classes, eye_scores)):
    print(f"  IC {idx:02d}: label={label:>8} score={score:.2f}")

print('Composantes oculaires retenues:', eye_components)
ica.exclude = eye_components

# Conserver une copie des epochs avant ICA pour comparaison
epochs_before_ica = epochs_clean.copy()

# 4. Appliquer la solution ICA sur le signal complet pre-ICA
raw_clean = ica.apply(raw_pre_ica.copy())
print('ICA appliquee au signal integral. Composantes exclues:', ica.exclude)

# Sauvegarder la version Raw nettoyee dans derivatives/preproc
deriv_root = bids_path.root / 'derivatives' / 'preproc'
processed_bids = bids_path.copy().update(
    root=deriv_root,
    datatype='eeg',
    suffix='processed',
    extension='.fif',
    processing='clean'
)
processed_bids.fpath.parent.mkdir(parents=True, exist_ok=True)
raw_clean.save(processed_bids.fpath, overwrite=True)
print('Fichier pretraite sauvegarde :', processed_bids.fpath)

# Conserver aussi la version nettoyee des epochs pour les analyses suivantes
epochs_clean = ica.apply(epochs_clean.copy())


In [ ]:
ica.plot_components()

In [ ]:
# TODO: ajustez l'ICA / ICLabel
nb_composantes = 'A_COMPLETER'    # ex. 25
method = 'fastica'                # ou 'picard', 'infomax', etc.
seuil_oeil = 'A_COMPLETER'        # ex. 0.5
seuil_muscle = 'A_COMPLETER'      # ex. 0.7

if 'A_COMPLETER' in [nb_composantes, seuil_oeil, seuil_muscle]:
    raise ValueError('Renseignez le nombre de composantes et les seuils avant execution.')

ica_custom = mne.preprocessing.ICA(n_components=int(nb_composantes), method=method, random_state=51)
ica_custom.fit(epochs_clean)
labels_custom = iclabel_label_components(epochs_clean, ica_custom)
classes_custom = np.array(labels_custom['labels'])
probabilities_custom = labels_custom.get('probabilities', [])
class_names_custom = labels_custom.get('classes', ['brain', 'muscle', 'eye', 'heart', 'line_noise', 'channel_noise', 'other'])

if isinstance(probabilities_custom, list):
    probas_custom = np.array(probabilities_custom)
else:
    probas_custom = np.asarray(probabilities_custom)

if probas_custom.ndim != 2 or probas_custom.shape[1] != len(class_names_custom):
    probas_custom = np.zeros((len(classes_custom), len(class_names_custom)))
    for idx, label in enumerate(classes_custom):
        if label in class_names_custom:
            probas_custom[idx, class_names_custom.index(label)] = 1.0

idx_eye = class_names_custom.index('eye')
idx_muscle = class_names_custom.index('muscle')
exclude_custom = np.where((classes_custom == 'eye') & (probas_custom[:, idx_eye] > float(seuil_oeil)))[0].tolist()
exclude_custom += np.where((classes_custom == 'muscle') & (probas_custom[:, idx_muscle] > float(seuil_muscle)))[0].tolist()

print('Composantes selectionnees:', exclude_custom)
ica_custom.exclude = exclude_custom
ica_custom.plot_properties(epochs_clean)


#### Bloc Type 3 — Pour aller plus loin
Évaluez l'impact de l'ICA sur la PSD ou les ERP, comparez avec et sans exclusion.

In [ ]:
before = epochs_before_ica.average()
after = epochs_clean.average()

fig, ax = plt.subplots(figsize=(8, 4))
before.plot(axes=ax, show=False, spatial_colors=True)
ax.set_title('ERP avant ICA (epochs selectionnees)')
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
after.plot(axes=ax, show=False, spatial_colors=True)
ax.set_title('ERP apres ICA (epochs selectionnees)')
plt.show()


### Étape 2.4 — Deuxième passe AutoReject et sauvegarde
Nous réappliquons AutoReject après ICA puis sauvegardons les données prêtes pour les analyses (Module 7).

#### Bloc Type 1 — Méthode de base
AutoReject rapide sur les epochs ICA-clean et sauvegarde au format `.fif`.

In [ ]:
epochs_post_ica = auto_reject.transform(epochs_clean)
print('Dimensions finales:', epochs_post_ica.get_data().shape)

epochs_bids = bids_path.copy().update(
    root=bids_path.root / 'derivatives' / 'preproc',
    datatype='eeg',
    suffix='epo',
    extension='.fif',
    processing='clean'
)
epochs_bids.fpath.parent.mkdir(parents=True, exist_ok=True)
epochs_post_ica.save(epochs_bids.fpath, overwrite=True)
print('Epochs sauvegardees dans', epochs_bids.fpath)


#### Bloc Type 2 — À vous de compléter
Ajoutez d'autres étapes (re-référence finale, interpolation de canaux, égalisation des conditions) avant la sauvegarde.

In [ ]:
# TODO: inserer des etapes additionnelles avant la sauvegarde
# Exemple: interpolation de capteurs marques comme mauvais
# raw_filt.interpolate_bads(reset_bads=True)
# epochs_final = ...

# chemin_epochs_personnalise = chemin_epochs.with_name(chemin_epochs.name.replace('clean', 'custom'))
# epochs_final.save(chemin_epochs_personnalise, overwrite=True)


#### Bloc Type 3 — Pour aller plus loin
Générez un rapport MNE (`mne.Report`), calculez des métriques de qualité, documentez vos décisions (Module 5).

In [ ]:
from mne.report import Report

report = Report(title='Fivepoint EEG — Pipeline PSY2007')
report.add_raw(raw, title='Brut', psd=True)
report.add_epochs(epochs_post_ica, title='Epochs nettoyees')
report_path = epochs_bids.fpath.parent / 'fivepoint_pipeline_report.html'
report.save(report_path, overwrite=True, open_browser=False)
print('Rapport genere:', report_path)


## 3. Factoriser le pipeline pour tous les fichiers

Objectifs : créer des fonctions modulaires qui chargent, nettoient et sauvegardent automatiquement l'ensemble des enregistrements BIDS.

In [ ]:
def pretraiter_sujet(subject, session, task, run, root=root_bids, line_freq=60):
    bids_path = BIDSPath(root=root, subject=subject, session=session,
                         task=task, run=run, suffix='eeg', extension='.edf')
    raw = read_raw_bids(bids_path, preload=True)

    raw.notch_filter(freqs=[line_freq])
    raw.filter(l_freq=1.0, h_freq=40.0, fir_design='firwin')
    raw.set_eeg_reference('average', projection=True)

    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, on_missing='warn')

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id, tmin=-0.2, tmax=0.8,
                        baseline=(None, 0), preload=True)

    auto_reject_local = AutoReject(random_state=42, verbose=False)
    auto_reject_local.fit(epochs)
    reject_log = auto_reject_local.get_reject_log(epochs)
    epochs_clean = epochs[~reject_log.bad_epochs].copy()

    ica = ICA(n_components=20, method='fastica', random_state=97)
    ica.fit(epochs_clean)
    labels = iclabel_label_components(epochs_clean, ica)
    classes = np.array(labels['labels'])
    probabilities = labels.get('probabilities', [])
    class_names = labels.get('classes', ['brain', 'muscle', 'eye', 'heart', 'line_noise', 'channel_noise', 'other'])

    if isinstance(probabilities, list):
        probas = np.array(probabilities)
    else:
        probas = np.asarray(probabilities)

    if probas.ndim != 2 or probas.shape[1] != len(class_names):
        probas = np.zeros((len(classes), len(class_names)))
        for idx, label in enumerate(classes):
            if label in class_names:
                probas[idx, class_names.index(label)] = 1.0

    index_eye = class_names.index('eye')
    eye_scores = probas[:, index_eye]
    eye_components = [idx for idx, label in enumerate(classes) if label == 'eye' and eye_scores[idx] >= 0.7]
    ica.exclude = eye_components

    raw_clean = ica.apply(raw.copy())
    epochs_clean = ica.apply(epochs_clean.copy())
    epochs_post_ica = auto_reject_local.transform(epochs_clean)

    deriv_root = root / 'derivatives' / 'preproc'

    processed_bids = bids_path.copy().update(
        root=deriv_root,
        datatype='eeg',
        suffix='processed',
        extension='.fif',
        processing='clean'
    )
    processed_bids.fpath.parent.mkdir(parents=True, exist_ok=True)
    raw_clean.save(processed_bids.fpath, overwrite=True)

    epochs_bids = bids_path.copy().update(
        root=deriv_root,
        datatype='eeg',
        suffix='epo',
        extension='.fif',
        processing='clean'
    )
    epochs_bids.fpath.parent.mkdir(parents=True, exist_ok=True)
    epochs_post_ica.save(epochs_bids.fpath, overwrite=True)
    return epochs_bids.fpath

# Traitement d'exemple
if root_bids.exists() and subjects != ['A_COMPLETER']:
    for sub in subjects:
        resultat = pretraiter_sujet(sub, session=session, task=task, run=run)
        print('Sujet', sub, 'traite ->', resultat)


### Bloc Type 2 — À vous de compléter
Généralisez la fonction : paramètres pour le bandpass, gestion d'erreurs (fichier manquant), ajout de logs. Utilisez les pratiques du Module 3 (fonctions) et du Module 0 (gestion de fichiers).

In [ ]:
# TODO: ameliorer la fonction pipeline
def pretraiter_sujet_custom(subject, session, task, run,
                            root, freqs_notch, bandpass, reference,
                            tmin, tmax, auto_reject_params, n_components):
    '''Completer cette fonction pour appliquer un pipeline parametrable.'''
    raise NotImplementedError('A implementer par les etudiants')

# TODO: boucle sur tous les sujets / sessions disponibles
# for subject in subjects:
#     try:
#         pretraiter_sujet_custom(...)
#     except FileNotFoundError:
#         print('Sujet', subject, 'introuvable. Verifiez la structure BIDS.')


### Bloc Type 3 — Pour aller plus loin
Automatisez la génération de rapports, l'export CSV des métriques, ou l'intégration avec un pipeline de ML (Module 7).

In [ ]:
# Exemple: aggregation des chemins d'epochs nettoyees
from collections import defaultdict

resultats = defaultdict(list)
if root_bids.exists() and subjects != ['A_COMPLETER']:
    for subject in subjects:
        chemin = root_bids / 'derivatives' / 'preproc'
        resultats['subject'].append(subject)
        resultats['epochs_path'].append(str(chemin))

pd.DataFrame(resultats)


## 4. Synthèse et prochaines étapes

Vous avez maintenant :
1. Exploré la structure BIDS et visualisé les données brutes
2. Rejoué un pipeline complet de nettoyage (filtrage, AutoReject, ICA, sauvegarde)
3. Factorisé le pipeline en fonctions réutilisables pour traiter tous les fichiers

Prochaine étape: analyse temps-fréquence ou extraction de caractéristiques (Modules 6 et 7) tout en documentant chaque paramètre pour assurer la reproductibilité. Bon travail !